In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy 
import gymnasium as gym
from ignorav2 import CustomTaxiEnv
from gymnasium.wrappers import TimeLimit

In [ ]:
import gymnasium as gym

# Create the environment
env = gym.make('Taxi-v3', render_mode= None)

# Reset the environment
obs, info = env.reset()

# Initialize the PPO model
model = PPO("MlpPolicy", env, verbose=1)

# Train the model (this will replace the loop with random actions)
model.learn(total_timesteps=1000000)  # Define the number of timesteps to train the model

# Save the trained model
model.save("ppo_taxi_model")

del model

model = PPO.load("ppo_taxi_model.zip")

# After training, you can test the model by running it in the environment
obs, info = env.reset()


mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes = 100, deterministic = True)
print(f"Mean reward {mean_reward}; Standard Deviation reward {std_reward}")

**Environment Customization** - Classe criada en ignorav2 

In [ ]:
# grid_size= (10,10)
# possible_locations = [(0, 0),  # Red
#             (0, grid_size[1] - 1),  # Green
#             (grid_size[0] - 1, 0),  # Yellow
#             (grid_size[0] - 1, grid_size[1] - 1)]  # Blue


# env = CustomTaxiEnv(grid_size=grid_size, possible_locations=possible_locations)

# # Wrap the environment with TimeLimit to enforce episode length
# env = TimeLimit(env, max_episode_steps=200)

# # Reset the environment
# obs, info = env.reset()

# # Initialize the PPO model
# model = PPO("MlpPolicy", env, verbose=1)

# # Train the model (this will replace the loop with random actions)
# model.learn(total_timesteps=1000000)  # Define the number of timesteps to train the model

# # Save the trained model
# model.save("ppo_taxi_model")

# del model

# model = PPO.load("ppo_taxi_model")

# # After training, you can test the model by running it in the environment
# obs, info = env.reset()


# mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes = 100, deterministic = True)
# print(f"Mean reward {mean_reward}; Standard Deviation reward {std_reward}")

In [ ]:

# # Run a loop to test the trained model
# num_steps = 100  # Define the number of steps for testing
# for step in range(num_steps):
    
#     action, _states = model.predict(obs)  # Use the trained model to predict actions
#     print(action)
#     action = int(action)
#     obs, reward, done, truncated, info = env.step(action)  # Step through the environment

#     print(f"Step {step + 1}:")
#     print(f"Action taken: {action}")
#     print(f"New state: {obs}")
#     print(f"Reward: {reward}")
#     print(f"Done: {done}")
#     print(f"Truncated: {truncated}\n")
#     env.render()

#     if done or truncated:
#         print("Episode finished.")
#         break

# # Close the environment
# env.close()

**TransformReward wrapper**

No ambiente original, recompensas encontram-se definidas da seguinte forma:

<img src="media/rewards.png" width="500">

In [ ]:
from gymnasium.wrappers import TransformReward

def custom_reward_function(reward):
    if reward == 10:  # Successful pickup
        return reward + 5  # Extra reward for pickups
    elif reward == 20:  # Successful dropoff
        return reward + 10  # Extra reward for drop-offs
    else:
        return reward

env = gym.make("Taxi-v3")
env = TransformReward(env, custom_reward_function)


# Reset the environment
obs, info = env.reset()

# Initialize the PPO model
model = PPO("MlpPolicy", env, verbose=1)

# Train the model (this will replace the loop with random actions)
model.learn(total_timesteps=1000000)  # Define the number of timesteps to train the model

# Save the trained model
model.save("ppo_taxi_model")

del model

model = PPO.load("ppo_taxi_model")

# After training, you can test the model by running it in the environment
obs, info = env.reset()


mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes = 100, deterministic = True)
print(f"Mean reward {mean_reward}; Standard Deviation reward {std_reward}")

In [ ]:
# # Run a loop to test the trained model
# num_steps = 100  # Define the number of steps for testing
# for step in range(num_steps):
    
#     action, _states = model.predict(obs)  # Use the trained model to predict actions
#     print(action)
#     action = int(action)
#     obs, reward, done, truncated, info = env.step(action)  # Step through the environment

#     print(f"Step {step + 1}:")
#     print(f"Action taken: {action}")
#     print(f"New state: {obs}")
#     print(f"Reward: {reward}")
#     print(f"Done: {done}")
#     print(f"Truncated: {truncated}\n")
#     env.render()

#     if done or truncated:
#         print("Episode finished.")
#         break

# # Close the environment
# env.close()

In [ ]:
import gymnasium as gym
from gymnasium.envs.toy_text.taxi import TaxiEnv

class Taxi(TaxiEnv):
    def __init__(self):
        super().__init__()

    def step(self, action):
        # Call the parent step method to get the original observation, reward, etc.
        observation, reward, done, truncated, info = super().step(action)

        # Decode the observation into meaningful state variables
        taxi_row, taxi_col, passenger, destination = self.decode(observation)

        # Custom reward logic for movement actions (0-3)
        if action in [0, 1, 2, 3]:  # North, South, East, West
            reward -= 1  # Default step penalty for movement

        # Custom reward logic for pickup action (4)
        elif action == 4:  # Pickup
            if passenger != 4 and self.locs[passenger] == (taxi_row, taxi_col):  # Valid pickup location
                reward += 10 # Reward for valid pickup
            else:
                reward -= 5 # Penalty for invalid pickup (wrong location or already delivered)

        # Custom reward logic for dropoff action (5)
        elif action == 5:  # Dropoff
            if passenger == 4 and self.locs[destination] == (taxi_row, taxi_col):  # Valid dropoff location
                reward += 20 # Reward for valid dropoff
            else:
                reward -= 10  # Penalty for invalid dropoff (wrong location or wrong passenger)

        return observation, reward, done, truncated,info

In [ ]:
import gymnasium as gym
# import TimeLimit from gymnasium.wrappers
from gymnasium.wrappers import TimeLimit

# Create the environment
env = Taxi()
env = TimeLimit(env, max_episode_steps=200)

# Reset the environment
obs, info = env.reset()

# Initialize the PPO model
model = PPO("MlpPolicy", env, verbose=1)

# Train the model (this will replace the loop with random actions)
model.learn(total_timesteps=1000000)  # Define the number of timesteps to train the model

# Save the trained model
model.save("ppo_taxi_model")

del model

model = PPO.load("ppo_taxi_model.zip")

# After training, you can test the model by running it in the environment
obs, info = env.reset()


mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes = 100, deterministic = True)
print(f"Mean reward {mean_reward}; Standard Deviation reward {std_reward}")

In [ ]:
# Run a loop to test the trained model
num_steps = 100  # Define the number of steps for testing
for step in range(num_steps):
    
    action, _states = model.predict(obs)  # Use the trained model to predict actions
    print(action)
    action = int(action)
    obs, reward, done, truncated, info = env.step(action)  # Step through the environment

    print(f"Step {step + 1}:")
    print(f"Action taken: {action}")
    print(f"New state: {obs}")
    print(f"Reward: {reward}")
    print(f"Done: {done}")
    print(f"Truncated: {truncated}\n")
    #env.render()

    if done or truncated:
        print("Episode finished.")
        break

# Close the environment
env.close()

In [3]:
from stable_baselines3.common.vec_env import DummyVecEnv
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
def make_env():
    env = gym.make("Taxi-v3")
    env = TimeLimit(env, max_episode_steps=200)
    return env

env = DummyVecEnv([make_env for _ in range(4)])

In [4]:
print(env)